# 04 — Model Evaluation
Evaluate an unsupervised detector with anomaly rate, score distribution and a controlled stress test.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
ROOT = Path.cwd().parent
RAW = ROOT/"data"/"raw"/"water_quality_raw.csv"
PROC = ROOT/"data"/"processed"
MODELS = ROOT/"models"

import joblib
model=joblib.load(MODELS/"anomaly_detector.pkl"); scaler=joblib.load(MODELS/"scaler.pkl")
meta=json.loads((MODELS/"model_metadata.json").read_text()); features=meta["features"]
df=pd.read_csv(PROC/"water_quality_clean.csv"); X=scaler.transform(df[features])
pred=model.predict(X); scores=model.decision_function(X)
print("Observed anomaly rate:",round((pred==-1).mean()*100,2),"%")


In [ ]:
plt.hist(scores,bins=40); plt.axvline(0,linestyle="--"); plt.title("Decision Score Distribution"); plt.xlabel("Decision score"); plt.ylabel("Frequency"); plt.tight_layout(); plt.show()


In [ ]:
stress=df[features].sample(min(100,len(df)),random_state=RANDOM_STATE).copy()
if "ph" in stress: stress["ph"]=np.clip(stress["ph"]-1.5,0,None)
if "turbidity" in stress: stress["turbidity"]*=8
if "tds" in stress: stress["tds"]*=2.5
sp=model.predict(scaler.transform(stress))
print("Stress-test anomaly rate:",round((sp==-1).mean()*100,2),"%")


In [ ]:
print("Use validated/labeled samples to assess false alarms when available. Do not interpret anomaly rate alone as model quality.")
